# SciPy Sparse and Optimize Modules

## What is SciPy?

SciPy is a general purpose open source scientific computing library for python, which includes algorithms for optimization, linear algebra problems, differentiation and integration, and many other classes of problems. SciPy wraps LAPACK linear algbra functions iternally to provide an optimized implementation, and it extends many basic features that numpy offers by adding new data structures such as sparse matrices and accompanying algorithms. The sparse module specifically focusses on sparse matrix problems, including iterative algorithms for solving least squares systems as well as low-rank algorithms such as low-rank SVD. 

## Stakeholders and Development

Many engineers and researchers use SciPy for its general and extremely large set of easy-to-use algorithms. SciPy is written in python but wraps LAPACK for its core operations, so the library does not sacrafice much performance by using python. The project is maintained primarily by a set of volunteers, but the institutions Quansight and Los Alamos National Laboratory employ contributors as well. (https://scipy.org/about/)

The project began in 2001, has 15.2k stars, has 246 contributors in the past year, and has about 45 commits a week. People communicate through github issues and on the SciPy developer forum (https://discuss.scientific-python.org/c/contributor/scipy/32). 

## Example Code Snippet (from https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.cg.html)

This short python script solves a system Ax=b where A is symmetric and positive definite using the conjugate gradient method.

In [1]:
import numpy as np
from scipy.sparse import csc_array
from scipy.sparse.linalg import cg

P = np.array([[4, 0, 1, 0],
              [0, 5, 0, 0],
              [1, 0, 3, 2],
              [0, 0, 2, 4]])
A = csc_array(P)
b = np.array([-1, -0.5, -1, 2])
print(f'For array A:\n{A}\nand vector b:\n{b}')
x, exit_code = cg(A, b, atol=1e-5)
print(f'Output: {x}')
print(np.allclose(A.dot(x), b))

For array A:
<Compressed Sparse Column sparse array of dtype 'int64'
	with 8 stored elements and shape (4, 4)>
  Coords	Values
  (0, 0)	4
  (2, 0)	1
  (1, 1)	5
  (0, 2)	1
  (2, 2)	3
  (3, 2)	2
  (2, 3)	2
  (3, 3)	4
and vector b:
[-1.  -0.5 -1.   2. ]
Output: [ 5.03611909e-17 -1.00000000e-01 -1.00000000e+00  1.00000000e+00]
True


## SciPy Design Choices in scipy.sparse and scipy.optimize

* Functions are wrapped in classes that initialize most values lazily (such as the gradient). Makes these wrappers very general while being optimized.
* Most function wrappers are also memoized, small overhead and many algorithms repeatedly call functions with the same inputs.
* Low-level linear algebra operations involve callbacks to a LAPACK library (OpenBLAS by default) or numpy, which itself calls OpenBLAS operaitons internally.
  * When SciPy bypasses numpy and calls a low-level BLAS operation directly it likely means the numpy overhead wouldn't be negligible, which can be the case if the operation is speed bound and not memory bound. So for level-1 and 2 BLAS operations on very large matrices it doesn't matter as much to bypass numpy, as the overhead doesn't matter much compared to the total time spent and memory-speed limitations.
* SciPy functions often have many fallbacks if certain inputs can be estimated from others. For example in Newton-CG, one is able to provide a function to compute applying the hessian to a vector which is much faster, but SciPy will estimate this using the jacobian if this is not provided.

## Project Idea: Comparing Newton-CG and BFGS

My idea for a project is to comprae the speed and convergence rates of both Newton-CG and BFGS when applied to different non-linear systems which must be minimized or maximized. Newton-CG should converge faster, but is much more expensive due to needing to perform CG at each step to approximate the inverse Hessian. BFGS, since it is a quasi-Newton method, originally behaves like gradient descent (assuming the initial guess for the Hessian is the identity which is very common), and progressively updates its inverse Hessian approximation at each step using a cheap iteration function. This makes it converge much slower but each iteration is much easier to process, so neither method is generally superior to the other. Newton-CG also only requires Hessian-vector products while BFGS requires storing the full inverse Hessian, so this may impact performance as well if one can provide a simpler Hessian-vector product function.

## Overview of Newton-CG and BFGS

The core principle of both algorithms comes from the second order Taylor-approximation of a multivariable function, where we want to find when the gradient of a function $f$ is 0:

$$f(\mathbf{x}+\mathbf{s}) \approx f(\mathbf{x})+\nabla f(\mathbf{x})^T\mathbf{s}+\frac{1}{2}\mathbf{s}^T H\mathbf{s}$$

$$\nabla_{\mathbf{s}}f(\mathbf{x}+\mathbf{s})=\nabla f(\mathbf{x})+H\mathbf{s}=\mathbf{0}\quad\Longrightarrow\quad H\mathbf{s}=-\nabla f(\mathbf{x})$$

This means that given a way to compute the gradient and hessian of a function (which SciPy can approximate if no hessian function is given), one wants to solve the system $Hs = \nabla^2 f(\mathbf{x})s = -\nabla f(\mathbf{x})$. 

For Newton-CG, one uses the conjugate gradient method to iteratively estimate a solution to this, since the hessian matrix is symmetric and positive definite. Conjugate gradient works by solving the system in a growing Krylov basis formed by vectors which are orthogonal when using the inner product defined by $\langle u, v \rangle = u^THv$. This is also refered to as the vectors being conjugate to each other. Given an initial guess, the first iteration of conjugate gradient uses the gradient descent vector which moves toward the minimum of the quadratic function $q(\mathbf{x})=\frac{1}{2}\mathbf{x}^T A\mathbf{x}-\mathbf{b}^T\mathbf{x}$. This is $-\nabla q(\mathbf{x})=\mathbf{b} - A\mathbf{x}$, which is the residual $r_1$ of the first step. Future vectors are produced by orthogonalizing future residuals against previous ones using the custon inner product to produce an update direction $p_k$, and updating the guess to be the $x_{k+1} = x_k + \alpha p_k$ where $\alpha$ is chosen to make $r_k^Tp_k = 0$. $\alpha$ is chosen this way because the residual also happens to be the gradient descent vector, and one wants this to be 0 to find the minimum of the quadratic function. Given the rules for for forming $p_k$, one can show that this implies $r_k^Tr_k = 0$, so residuals are orthogonal under the euclidian inner product and the vectors $p_k$ are orthogonal under the A-inner product. This results in a very simple algorithm to update $r_k$ and $p_k$, since it can be shown that because of thse properties future $p_k$ vectors only need to be orthogonalized against the immediately preceding $p_{k-1}$ vector. Once one obtains the A-orthogonal $p_k$ basis, one can solve Ax = b by expressing x in this basis and multiplying on the left by $p_k^T$. Since $p_k^TAx = p_k^TA \sum_i {k_ip_i} = k_i p_k^TAp_k = p_k^Tb$, this provides an easy way to solve for basis coefficients $k_i$, providing a solution. 

For BFGS, one instead starts with an initial inverse Hessian guess and updates it at each iteration using the formula:

$$H_{k+1}=\left(I-\rho_k\mathbf{s}_k\mathbf{y}_k^T\right)H_k\left(I-\rho_k\mathbf{y}_k\mathbf{s}_k^T\right)+\rho_k\mathbf{s}_k\mathbf{s}_k^T,\qquad \rho_k=\frac{1}{\mathbf{y}_k^T\mathbf{s}_k}$$

$$\text{where }\mathbf{s}_k=\mathbf{x}_{k+1}-\mathbf{x}_k\text{ and }\mathbf{y}_k=\nabla f(\mathbf{x}_{k+1})-\nabla f(\mathbf{x}_k)$$

This formula can be derived by enforcing symmetry of the hessian and using the secant forumla for the Hessian, relating it to the gradient as $B_{k+1}\mathbf{s}_k=\mathbf{y}_k=\nabla f(\mathbf{x}_{k+1})-\nabla f(\mathbf{x}_k)$, where $B_{k+1}$ is the hessian approximation. The line search method used in both Newton-CG and BFGS ensures the curvature condition, which also ensures the hessian remains positive definite during this iterative process.

## Newton-CG Code Example

In [3]:
import numpy as np
from scipy.optimize import minimize, rosen, rosen_der, rosen_hess

x0 = np.array([-1.2, 1.0])

result = minimize(
    rosen,
    x0,
    method="Newton-CG",
    jac=rosen_der,
    hess=rosen_hess
)

print("Minimum:", result.x)
print("Function value:", result.fun)
print("Converged:", result.success)

Minimum: [0.9999826  0.99996514]
Function value: 3.0314331378333585e-10
Converged: True


## BFGS Code Example

In [4]:
x0 = np.array([-1.2, 1.0])

result = minimize(
    rosen,
    x0,
    method="BFGS",
    jac=rosen_der
)

print("Minimum:", result.x)
print("Function value:", result.fun)
print("Converged:", result.success)

Minimum: [0.99999997 0.99999995]
Function value: 2.5353057938593763e-15
Converged: True
